# ARIMA semanal: ejecución verificable
Se conserva el método de `ArimaPredictor.py`: suma semanal por sucursal-producto (`W`, domingo), ceros en huecos interiores, filtros de longitud (20 semanas totales, 10 train, 2 test), corte 2026-01-01 por etiqueta semanal, ADF para d y búsqueda p/q de 0 a 3 por AIC. No se eliminan series por su porcentaje de ceros.

Se pronostica todo el test sin incorporar observaciones del test al ajuste. Las métricas globales se calculan concatenando valores reales y pronosticados de las series modeladas; no se promedian métricas por serie. La primera semana de test termina el 2026-01-04 e incluye días de diciembre: el corte se aplica a la etiqueta de la semana, no divide esa semana en dos. Las semanas de los extremos pueden estar incompletas.

La granularidad coincide con ML, pero el horizonte de ARIMA es multisemana desde un origen fijo, mientras ML realiza evaluación secuencial de una semana. La cobertura también depende de los filtros ARIMA. No interpretar ambas tablas como una competencia estrictamente emparejada.

In [1]:
import sys
import pandas as pd

# 1. Le decimos a Python que busque módulos en la carpeta principal
sys.path.append("../") 

# 2. Importamos tu clase desde la ruta src/Modelos/ArimaPredictor.py
from src.Modelos.ArimaPredictor import ArimaPredictor

# 3. Cargar los datos
df = pd.read_csv("../datasets/dataset_maestro_dashboard.csv")

# 4. Instanciar y evaluar
arima_model = ArimaPredictor(df)
metricas_arima = arima_model.entrenar_y_evaluar()

print("\nResultados ARIMA (Línea Base):")
print(metricas_arima)

Iniciando entrenamiento ARIMA (Iteracion por Sucursal y Producto)...


Resumen de series ARIMA: {'potenciales': 266, 'encontradas': 240, 'descartadas': 111, 'fallidas': 0, 'modeladas': 129}
Entrenamiento completado.

Resultados ARIMA (Línea Base):
{'Modelo': 'ARIMA (Optimizado ADF/AIC)', 'MAE': 0.2286279096877065, 'RMSE': np.float64(0.5675260069553762), 'R2': 0.20926149308163855}


In [2]:
from pathlib import Path
import json
import hashlib
import numpy as np

salida = Path('../resultados/semanal')
salida.mkdir(parents=True, exist_ok=True)
assert arima_model.conteo_series['encontradas'] == sum(
    arima_model.conteo_series[k] for k in ['descartadas', 'fallidas', 'modeladas'])
display(pd.DataFrame([arima_model.conteo_series]))
pd.DataFrame([metricas_arima]).to_csv(salida / 'metricas_arima.csv', index=False)
(salida / 'conteo_arima.json').write_text(json.dumps(arima_model.conteo_series, indent=2), encoding='utf-8')
detalle_arima = pd.DataFrame({'y_real': arima_model.y_real_totales,
                            'y_pred': arima_model.predicciones_totales})
detalle_arima['error'] = detalle_arima.y_real - detalle_arima.y_pred
detalle_arima['error_absoluto'] = detalle_arima.error.abs()
detalle_arima.to_csv(salida / 'predicciones_arima_concatenadas.csv', index=False)
print('Observaciones ARIMA evaluadas:', len(detalle_arima))

,potenciales,encontradas,descartadas,fallidas,modeladas
0,266,240,111,0,129


Observaciones ARIMA evaluadas: 3573
